In [3]:
import ast
import json
import re
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch

from sentence_transformers import SentenceTransformer

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    f1_score,
    hamming_loss,
    classification_report,
    precision_score,
    recall_score,
)


# ======================================================
# 1. Настройки
# ======================================================

# Укажи путь к своему размеченному файлу
CSV_PATH = Path(r"classification_labeled.csv")

LABELS_COLUMN = "labels"

TEXT_COLUMN_CANDIDATES = [
    "target_groups",
    "target_group",
    "target_group_text",
    "text",
]

MODEL_NAME = "deepvk/USER-base"

OUTPUT_DIR = Path("model_artifacts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

TEST_SIZE = 0.20
VALID_SIZE = 0.20

BATCH_SIZE = 128


# ======================================================
# 2. Справочник категорий
# ======================================================

CATEGORY_NAMES = {
    1: "Дети",
    2: "Подростки",
    3: "Школьники",
    4: "Студенты",
    5: "Молодежь",
    6: "Молодые специалисты",
    7: "Родители и семьи с детьми",
    8: "Многодетные семьи",
    9: "Семьи участников СВО",
    10: "Дети-сироты и дети без попечения родителей",
    11: "Люди с инвалидностью и ОВЗ",
    12: "Пожилые люди",
    13: "Ветераны",
    14: "Военнослужащие и участники СВО",
    15: "Люди в трудной жизненной ситуации",
    16: "Люди с зависимостями",
    17: "Люди без определенного места жительства",
    18: "Медицинские пациенты и люди с хроническими заболеваниями",
    19: "Мигранты и переселенцы",
    20: "Лица в местах лишения свободы",
    21: "Жители территорий и местных сообществ",
    22: "Жители сельских территорий",
    23: "Женщины",
    24: "Педагоги и наставники",
    25: "Специалисты социальной сферы",
    26: "Специалисты и руководители",
    27: "Ученые и эксперты",
    28: "Волонтеры и общественные активисты",
    29: "Представители НКО",
    30: "Предприниматели и самозанятые",
    31: "Культурные и творческие сообщества",
    32: "Спортсмены и участники ЗОЖ",
    33: "Туристы",
    34: "Читатели и библиотечная аудитория",
    35: "Экологические сообщества",
    36: "СМИ и медиа",
    37: "Представители национальных, этнокультурных и казачьих сообществ",
    38: "Широкая аудитория",
}

VALID_LABELS = sorted(CATEGORY_NAMES.keys())


# ======================================================
# 3. Вспомогательные функции
# ======================================================

def parse_labels(value):
    """
    Преобразует значение labels в список int.
    Например:
    "[1, 7, 11]" -> [1, 7, 11]
    "" -> []
    NaN -> []
    """
    if pd.isna(value):
        return []

    if isinstance(value, list):
        raw = value
    else:
        value = str(value).strip()

        if value == "" or value == "[]":
            return []

        try:
            raw = json.loads(value)
        except Exception:
            try:
                raw = ast.literal_eval(value)
            except Exception:
                return []

    if not isinstance(raw, list):
        return []

    result = []

    for item in raw:
        try:
            label = int(item)
            if label in VALID_LABELS:
                result.append(label)
        except Exception:
            continue

    return sorted(set(result))


def find_text_column(df):
    for col in TEXT_COLUMN_CANDIDATES:
        if col in df.columns:
            return col

    raise ValueError(
        f"Не найден столбец с текстом. "
        f"В CSV должен быть один из столбцов: {TEXT_COLUMN_CANDIDATES}"
    )


def make_safe_name(value):
    value = value.lower()
    value = value.replace("+", "plus")
    value = re.sub(r"[^a-zа-я0-9_]+", "_", value)
    value = re.sub(r"_+", "_", value)
    return value.strip("_")


def optimize_thresholds(y_true, y_proba, classes):
    """
    Подбор индивидуального порога классификации для каждой категории.

    Это важно при несбалансированных классах:
    для частых категорий порог может быть выше,
    для редких категорий — ниже.
    """
    thresholds = {}
    grid = np.arange(0.10, 0.91, 0.05)

    for idx, class_id in enumerate(classes):
        y_class_true = y_true[:, idx]

        # Если в validation нет положительных примеров класса,
        # не подбираем порог агрессивно.
        if y_class_true.sum() == 0:
            thresholds[int(class_id)] = 0.50
            continue

        best_threshold = 0.50
        best_f1 = -1.0

        for threshold in grid:
            y_class_pred = (y_proba[:, idx] >= threshold).astype(int)

            score = f1_score(
                y_class_true,
                y_class_pred,
                zero_division=0,
            )

            if score > best_f1:
                best_f1 = score
                best_threshold = threshold

        thresholds[int(class_id)] = float(best_threshold)

    return thresholds


def apply_thresholds(y_proba, classes, thresholds):
    """
    Применяет индивидуальные thresholds к вероятностям.
    """
    y_pred = np.zeros_like(y_proba, dtype=int)

    for idx, class_id in enumerate(classes):
        threshold = thresholds.get(int(class_id), 0.50)
        y_pred[:, idx] = (y_proba[:, idx] >= threshold).astype(int)

    return y_pred


def calculate_metrics(y_true, y_pred):
    """
    Считает основные метрики multi-label классификации.
    """
    return {
        "micro_precision": precision_score(
            y_true, y_pred, average="micro", zero_division=0
        ),
        "micro_recall": recall_score(
            y_true, y_pred, average="micro", zero_division=0
        ),
        "micro_f1": f1_score(
            y_true, y_pred, average="micro", zero_division=0
        ),
        "macro_f1": f1_score(
            y_true, y_pred, average="macro", zero_division=0
        ),
        "weighted_f1": f1_score(
            y_true, y_pred, average="weighted", zero_division=0
        ),
        "hamming_loss": hamming_loss(y_true, y_pred),
    }


def format_metrics(metrics):
    return "\n".join(
        [
            f"Micro precision: {metrics['micro_precision']:.4f}",
            f"Micro recall:    {metrics['micro_recall']:.4f}",
            f"Micro F1:        {metrics['micro_f1']:.4f}",
            f"Macro F1:        {metrics['macro_f1']:.4f}",
            f"Weighted F1:     {metrics['weighted_f1']:.4f}",
            f"Hamming loss:    {metrics['hamming_loss']:.4f}",
        ]
    )


# ======================================================
# 4. Загрузка данных
# ======================================================

print("=" * 80)
print("Загрузка данных")
print("=" * 80)

df = pd.read_csv(CSV_PATH)

TEXT_COLUMN = find_text_column(df)

if LABELS_COLUMN not in df.columns:
    raise ValueError(f"В CSV нет столбца `{LABELS_COLUMN}`")

df[TEXT_COLUMN] = df[TEXT_COLUMN].fillna("").astype(str)
df[LABELS_COLUMN] = df[LABELS_COLUMN].apply(parse_labels)

df = df[df[TEXT_COLUMN].str.strip() != ""].reset_index(drop=True)

texts = df[TEXT_COLUMN].tolist()
labels = df[LABELS_COLUMN].tolist()

print(f"Файл: {CSV_PATH}")
print(f"Текстовый столбец: {TEXT_COLUMN}")
print(f"Столбец с метками: {LABELS_COLUMN}")
print(f"Количество строк: {len(df):,}")
print(f"Пример labels: {labels[:5]}")


# ======================================================
# 5. Multi-hot encoding
# ======================================================

print("=" * 80)
print("Кодирование меток")
print("=" * 80)

mlb = MultiLabelBinarizer(classes=VALID_LABELS)
Y = mlb.fit_transform(labels)

print(f"Размер матрицы меток: {Y.shape}")
print(f"Количество категорий: {len(mlb.classes_)}")


# ======================================================
# 6. Распределение классов
# ======================================================

print("=" * 80)
print("Распределение классов")
print("=" * 80)

class_counts = Y.sum(axis=0)

class_distribution = pd.DataFrame(
    {
        "category_id": mlb.classes_,
        "category_name": [CATEGORY_NAMES[int(c)] for c in mlb.classes_],
        "count": class_counts,
        "share": class_counts / len(Y),
    }
).sort_values("count", ascending=False)

class_distribution_path = OUTPUT_DIR / "class_distribution.csv"

class_distribution.to_csv(
    class_distribution_path,
    index=False,
    encoding="utf-8-sig",
)

print(class_distribution)
print(f"Распределение классов сохранено: {class_distribution_path}")


# ======================================================
# 7. Расчет или загрузка эмбеддингов
# ======================================================

print("=" * 80)
print("Эмбеддинги")
print("=" * 80)

safe_csv_name = make_safe_name(CSV_PATH.stem)
safe_model_name = make_safe_name(MODEL_NAME)

EMBEDDINGS_PATH = OUTPUT_DIR / f"embeddings_{safe_csv_name}_{safe_model_name}.npy"

if EMBEDDINGS_PATH.exists():
    print("Загрузка сохраненных эмбеддингов...")

    X = np.load(EMBEDDINGS_PATH)

    if X.shape[0] != len(df):
        raise ValueError(
            f"Размер сохраненных эмбеддингов не совпадает с CSV: "
            f"{X.shape[0]} != {len(df)}. "
            f"Удали файл {EMBEDDINGS_PATH} и запусти скрипт заново."
        )

else:
    print("Расчет эмбеддингов...")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Устройство: {device}")

    embedder = SentenceTransformer(MODEL_NAME, device=device)

    X = embedder.encode(
        texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    np.save(EMBEDDINGS_PATH, X)

    print(f"Эмбеддинги сохранены: {EMBEDDINGS_PATH}")

print(f"Размер матрицы эмбеддингов: {X.shape}")


# ======================================================
# 8. Train / validation / test split
# ======================================================

print("=" * 80)
print("Разделение выборки")
print("=" * 80)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    Y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full,
    y_train_full,
    test_size=VALID_SIZE,
    random_state=RANDOM_STATE,
)

print(f"Train: {X_train.shape[0]:,}")
print(f"Valid: {X_valid.shape[0]:,}")
print(f"Test:  {X_test.shape[0]:,}")


# ======================================================
# 9. Обучение, оценка и сохранение моделей
# ======================================================

def train_evaluate_save_model(
    model_name,
    base_classifier,
    X_train,
    y_train,
    X_valid,
    y_valid,
    X_test,
    y_test,
    mlb,
    category_names,
    output_dir,
):
    print("=" * 80)
    print(f"Обучение модели: {model_name}")
    print("=" * 80)

    classifier = OneVsRestClassifier(
        base_classifier,
        n_jobs=-1,
    )

    classifier.fit(X_train, y_train)

    # --------------------------------------------------
    # Оценка с порогом 0.5
    # --------------------------------------------------

    test_proba_default = classifier.predict_proba(X_test)
    y_pred_default = (test_proba_default >= 0.50).astype(int)

    default_metrics = calculate_metrics(y_test, y_pred_default)

    # --------------------------------------------------
    # Подбор thresholds на validation
    # --------------------------------------------------

    print("Подбор индивидуальных thresholds...")

    valid_proba = classifier.predict_proba(X_valid)

    thresholds = optimize_thresholds(
        y_true=y_valid,
        y_proba=valid_proba,
        classes=mlb.classes_,
    )

    # --------------------------------------------------
    # Оценка с индивидуальными thresholds
    # --------------------------------------------------

    test_proba = classifier.predict_proba(X_test)

    y_pred_optimized = apply_thresholds(
        y_proba=test_proba,
        classes=mlb.classes_,
        thresholds=thresholds,
    )

    optimized_metrics = calculate_metrics(y_test, y_pred_optimized)

    # --------------------------------------------------
    # Classification report
    # --------------------------------------------------

    target_names = [
        f"{class_id}. {category_names[int(class_id)]}"
        for class_id in mlb.classes_
    ]

    report = classification_report(
        y_test,
        y_pred_optimized,
        target_names=target_names,
        zero_division=0,
    )

    # --------------------------------------------------
    # Сохранение результатов
    # --------------------------------------------------

    safe_name = make_safe_name(model_name)

    metrics_path = output_dir / f"metrics_{safe_name}.txt"
    model_path = output_dir / f"model_{safe_name}.joblib"
    thresholds_path = output_dir / f"thresholds_{safe_name}.json"

    metrics_text = f"""
MODEL: {model_name}
EMBEDDING_MODEL: {MODEL_NAME}
CSV_PATH: {CSV_PATH}
TEXT_COLUMN: {TEXT_COLUMN}
ROWS: {len(df)}
CATEGORIES: {len(mlb.classes_)}

============================================================
Metrics with default threshold = 0.50
============================================================

{format_metrics(default_metrics)}

============================================================
Metrics with optimized thresholds
============================================================

{format_metrics(optimized_metrics)}

============================================================
Optimized thresholds
============================================================

{json.dumps(thresholds, ensure_ascii=False, indent=2)}

============================================================
Classification report
============================================================

{report}
"""

    metrics_path.write_text(metrics_text, encoding="utf-8")

    with thresholds_path.open("w", encoding="utf-8") as f:
        json.dump(thresholds, f, ensure_ascii=False, indent=2)

    model_pack = {
        "classifier": classifier,
        "mlb": mlb,
        "thresholds": thresholds,
        "category_names": category_names,
        "embedding_model_name": MODEL_NAME,
        "text_column": TEXT_COLUMN,
        "labels_column": LABELS_COLUMN,
        "model_name": model_name,
    }

    joblib.dump(model_pack, model_path)

    print(metrics_text)
    print(f"Метрики сохранены: {metrics_path}")
    print(f"Thresholds сохранены: {thresholds_path}")
    print(f"Модель сохранена: {model_path}")

    result = {
        "model_name": model_name,

        "default_micro_precision": default_metrics["micro_precision"],
        "default_micro_recall": default_metrics["micro_recall"],
        "default_micro_f1": default_metrics["micro_f1"],
        "default_macro_f1": default_metrics["macro_f1"],
        "default_weighted_f1": default_metrics["weighted_f1"],
        "default_hamming_loss": default_metrics["hamming_loss"],

        "optimized_micro_precision": optimized_metrics["micro_precision"],
        "optimized_micro_recall": optimized_metrics["micro_recall"],
        "optimized_micro_f1": optimized_metrics["micro_f1"],
        "optimized_macro_f1": optimized_metrics["macro_f1"],
        "optimized_weighted_f1": optimized_metrics["weighted_f1"],
        "optimized_hamming_loss": optimized_metrics["hamming_loss"],

        "model_path": str(model_path),
        "metrics_path": str(metrics_path),
        "thresholds_path": str(thresholds_path),
    }

    return result


# ======================================================
# 10. Список сравниваемых моделей
# ======================================================

models = [
    {
        "name": "OneVsRest + SGDClassifier",
        "classifier": SGDClassifier(
            loss="log_loss",
            penalty="l2",
            alpha=1e-5,
            max_iter=2000,
            tol=1e-3,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
    },
    {
        "name": "OneVsRest + LogisticRegression",
        "classifier": LogisticRegression(
            solver="liblinear",
            penalty="l2",
            C=1.0,
            class_weight="balanced",
            max_iter=1000,
            random_state=RANDOM_STATE,
        ),
    },
    {
        "name": "OneVsRest + LinearSVC",
        "classifier": CalibratedClassifierCV(
            estimator=LinearSVC(
                C=1.0,
                class_weight="balanced",
                max_iter=5000,
                random_state=RANDOM_STATE,
            ),
            cv=3,
        ),
    },
    {
        "name": "OneVsRest + MLPClassifier",
        "classifier": MLPClassifier(
            hidden_layer_sizes=(256, 128),
            activation="relu",
            solver="adam",
            alpha=1e-4,
            batch_size=256,
            learning_rate_init=1e-3,
            max_iter=50,
            early_stopping=True,
            validation_fraction=0.1,
            random_state=RANDOM_STATE,
        ),
    },
]


# ======================================================
# 11. Запуск обучения и сравнения
# ======================================================

comparison_results = []

for model_config in models:
    result = train_evaluate_save_model(
        model_name=model_config["name"],
        base_classifier=model_config["classifier"],
        X_train=X_train,
        y_train=y_train,
        X_valid=X_valid,
        y_valid=y_valid,
        X_test=X_test,
        y_test=y_test,
        mlb=mlb,
        category_names=CATEGORY_NAMES,
        output_dir=OUTPUT_DIR,
    )

    comparison_results.append(result)


# ======================================================
# 12. Сохранение таблицы сравнения моделей
# ======================================================

comparison_df = pd.DataFrame(comparison_results)

comparison_path = OUTPUT_DIR / "model_comparison.csv"

comparison_df.to_csv(
    comparison_path,
    index=False,
    encoding="utf-8-sig",
)

print("=" * 80)
print("Итоговое сравнение моделей")
print("=" * 80)

print(comparison_df)

print(f"Таблица сравнения сохранена: {comparison_path}")


# ======================================================
# 13. Выбор лучшей модели по optimized_micro_f1
# ======================================================

best_row = comparison_df.sort_values(
    "optimized_micro_f1",
    ascending=False,
).iloc[0]

print("=" * 80)
print("Лучшая модель")
print("=" * 80)

print(f"Модель: {best_row['model_name']}")
print(f"Optimized Micro F1: {best_row['optimized_micro_f1']:.4f}")
print(f"Optimized Macro F1: {best_row['optimized_macro_f1']:.4f}")
print(f"Optimized Hamming loss: {best_row['optimized_hamming_loss']:.4f}")
print(f"Путь к модели: {best_row['model_path']}")


# ======================================================
# 14. Пример инференса на новых текстах
# ======================================================

def predict_with_saved_model(model_path, texts_to_predict):
    """
    Пример применения сохраненной модели к новым описаниям целевых групп.
    """

    model_pack = joblib.load(model_path)

    classifier = model_pack["classifier"]
    mlb = model_pack["mlb"]
    thresholds = model_pack["thresholds"]
    category_names = model_pack["category_names"]
    embedding_model_name = model_pack["embedding_model_name"]

    device = "cuda" if torch.cuda.is_available() else "cpu"

    embedder = SentenceTransformer(
        embedding_model_name,
        device=device,
    )

    X_new = embedder.encode(
        texts_to_predict,
        batch_size=64,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    proba = classifier.predict_proba(X_new)

    y_pred = apply_thresholds(
        y_proba=proba,
        classes=mlb.classes_,
        thresholds=thresholds,
    )

    label_ids_list = mlb.inverse_transform(y_pred)

    results = []

    for text, label_ids, row_proba in zip(texts_to_predict, label_ids_list, proba):
        label_ids = [int(x) for x in label_ids]

        labels_verbose = []

        for class_id in label_ids:
            class_index = list(mlb.classes_).index(class_id)

            labels_verbose.append(
                {
                    "category_id": class_id,
                    "category_name": category_names[class_id],
                    "score": float(row_proba[class_index]),
                    "threshold": float(thresholds.get(class_id, 0.50)),
                }
            )

        results.append(
            {
                "text": text,
                "label_ids": label_ids,
                "labels": labels_verbose,
            }
        )

    return results

Загрузка данных
Файл: classification_labeled.csv
Текстовый столбец: target_groups
Столбец с метками: labels
Количество строк: 129,964
Пример labels: [[7, 24], [2, 28, 29], [9], [1, 2, 38], [1, 2, 4, 5, 37]]
Кодирование меток
Размер матрицы меток: (129964, 38)
Количество категорий: 38
Распределение классов
    category_id  ...     share
0             1  ...  0.359938
1             2  ...  0.257756
4             5  ...  0.242383
3             4  ...  0.215490
10           11  ...  0.212613
2             3  ...  0.153966
11           12  ...  0.126412
6             7  ...  0.125404
25           26  ...  0.122603
14           15  ...  0.093664
20           21  ...  0.092541
7             8  ...  0.080992
23           24  ...  0.080661
22           23  ...  0.066557
27           28  ...  0.063102
17           18  ...  0.054923
9            10  ...  0.052268
12           13  ...  0.050629
28           29  ...  0.047690
26           27  ...  0.026992
8             9  ...  0.026861
18         

'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 4ddfe4b1-e1ab-481e-816e-94a952ad3552)')' thrown while requesting HEAD https://huggingface.co/deepvk/USER-base/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
Batches: 100%|██████████| 1016/1016 [03:07<00:00,  5.41it/s]


Эмбеддинги сохранены: model_artifacts/embeddings_classification_labeled_deepvk_user_base.npy
Размер матрицы эмбеддингов: (129964, 768)
Разделение выборки
Train: 83,176
Valid: 20,795
Test:  25,993
Обучение модели: OneVsRest + SGDClassifier
Подбор индивидуальных thresholds...

MODEL: OneVsRest + SGDClassifier
EMBEDDING_MODEL: deepvk/USER-base
CSV_PATH: classification_labeled.csv
TEXT_COLUMN: target_groups
ROWS: 129964
CATEGORIES: 38

Metrics with default threshold = 0.50

Micro precision: 0.5889
Micro recall:    0.9402
Micro F1:        0.7242
Macro F1:        0.5948
Weighted F1:     0.7728
Hamming loss:    0.0523

Metrics with optimized thresholds

Micro precision: 0.7795
Micro recall:    0.8672
Micro F1:        0.8210
Macro F1:        0.7189
Weighted F1:     0.8302
Hamming loss:    0.0276

Optimized thresholds

{
  "1": 0.5500000000000002,
  "2": 0.7500000000000002,
  "3": 0.6000000000000002,
  "4": 0.6500000000000001,
  "5": 0.6500000000000001,
  "6": 0.9000000000000002,
  "7": 0.75000

/home/jupyter/.local/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


Подбор индивидуальных thresholds...

MODEL: OneVsRest + MLPClassifier
EMBEDDING_MODEL: deepvk/USER-base
CSV_PATH: classification_labeled.csv
TEXT_COLUMN: target_groups
ROWS: 129964
CATEGORIES: 38

Metrics with default threshold = 0.50

Micro precision: 0.8946
Micro recall:    0.8549
Micro F1:        0.8743
Macro F1:        0.7832
Weighted F1:     0.8722
Hamming loss:    0.0180

Metrics with optimized thresholds

Micro precision: 0.8770
Micro recall:    0.8728
Micro F1:        0.8749
Macro F1:        0.7936
Weighted F1:     0.8749
Hamming loss:    0.0182

Optimized thresholds

{
  "1": 0.45000000000000007,
  "2": 0.45000000000000007,
  "3": 0.30000000000000004,
  "4": 0.45000000000000007,
  "5": 0.40000000000000013,
  "6": 0.1,
  "7": 0.3500000000000001,
  "8": 0.5000000000000001,
  "9": 0.1,
  "10": 0.3500000000000001,
  "11": 0.6000000000000002,
  "12": 0.3500000000000001,
  "13": 0.45000000000000007,
  "14": 0.40000000000000013,
  "15": 0.30000000000000004,
  "16": 0.7000000000000002

'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: f15dcdc6-b4ad-4988-9c2e-e018dd5b62ce)')' thrown while requesting HEAD https://huggingface.co/deepvk/USER-base/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


{
  "text": "дети с РАС и их родители",
  "label_ids": [
    1,
    7,
    11
  ],
  "labels": [
    {
      "category_id": 1,
      "category_name": "Дети",
      "score": 0.9997338652610779,
      "threshold": 0.45000000000000007
    },
    {
      "category_id": 7,
      "category_name": "Родители и семьи с детьми",
      "score": 0.998790442943573,
      "threshold": 0.3500000000000001
    },
    {
      "category_id": 11,
      "category_name": "Люди с инвалидностью и ОВЗ",
      "score": 0.999914288520813,
      "threshold": 0.6000000000000002
    }
  ]
}
{
  "text": "школьники и педагоги",
  "label_ids": [
    3,
    24
  ],
  "labels": [
    {
      "category_id": 3,
      "category_name": "Школьники",
      "score": 0.9977312684059143,
      "threshold": 0.30000000000000004
    },
    {
      "category_id": 24,
      "category_name": "Педагоги и наставники",
      "score": 0.9999613761901855,
      "threshold": 0.3500000000000001
    }
  ]
}
{
  "text": "пожилые люди в трудной

In [ ]:
example_texts = [
    "дети с РАС и их родители",
    "школьники и педагоги",
    "пожилые люди в трудной жизненной ситуации",
    "жители сельских территорий",
    "волонтеры и представители НКО",
]

best_model_path = best_row["model_path"]

print("=" * 80)
print("Пример предсказаний лучшей модели")
print("=" * 80)

predictions = predict_with_saved_model(
    model_path=best_model_path,
    texts_to_predict=example_texts,
)

for item in predictions:
    print(json.dumps(item, ensure_ascii=False, indent=2))